In [ ]:
import logging
from IPython.display import Markdown
from library.circuitry import Circuitry
from library.magic_state_cultivation import MagicStateCultivation
from library.common import Pauli
from library.qubit_array import QubitArray
from library.steane_code.patch import SteaneCodePatch
from library.surface_code.patch import SurfaceCodePatch
from utils.simulation.stim import simulate, sample

logging.basicConfig(level=logging.ERROR)

In [ ]:
TARGET_DISTANCE = 7

SUPERDENSE_ROUNDS = 3
TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

In [ ]:
FILEROOT = "generated/hirano-magic-state-cultivation-layout1"
scenarios: dict[str, Circuitry] = dict()
point: int = 0

In [ ]:
if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

minimum_anchoring = 1 + int(TARGET_DISTANCE == 7)
TARGET_ANCHOR = (minimum_anchoring, minimum_anchoring)

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits, clifford=True)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.S)

msc.append_preparation(circuitry)

circuitry.append_observable(0, "Y_OBSERVABLE_PREPARED", msc.steane.logical(Pauli.Y))

circuitry.detectors_report()

scenario = "Prepared"
scenarios[scenario] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.prepared")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})"))

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits, clifford=True)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.S)

msc.append_preparation(circuitry)
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(circuitry, rnd)

msc.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)

circuitry.append_observable(0, "Y_OBSERVABLE_SUPERDENSE", msc.steane.logical(Pauli.Y))

circuitry.detectors_report()

scenario = f"SDCx{SUPERDENSE_ROUNDS}"
scenarios[scenario] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.superdense")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})"))

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits, clifford=True)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.S)

msc.append_preparation(circuitry)
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(circuitry, rnd)
msc.append_cultivation(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)

circuitry.append_observable(0, "Y_OBSERVABLE_CULTIVATION", msc.steane.logical(Pauli.Y))

circuitry.detectors_report()

scenario = "Double-checked"
scenarios[scenario] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.double-check-s")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})"))

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits, clifford=True)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.S)

msc.append_preparation(circuitry)
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(circuitry, rnd)
msc.append_cultivation(circuitry)
msc.append_teleportation(circuitry, TELEPORT_ROUNDS)

msc.annotate_detectors(
    circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS
)

circuitry.append_observable(
    0,
    "Y_OBSERVABLE_TELEPORTED",
    msc.source.logical(Pauli.Y),
    *[
        "JCT0:Z0",
        "JCT0:Z1",
        "JCT0:Z2",
        "STN:TPT0:XB",
        "STN:TPT1:XB",
        "STN:TPT2:XB",
        "STN:DST:X1",
        "STN:DST:X5",
        "STN:DST:X6",
    ],
)

circuitry.detectors_report()

scenario = "Teleported"
scenarios[scenario] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.teleported")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})"))

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits, clifford=True)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic, injection=SteaneCodePatch.Injection.S)

msc.append_preparation(circuitry)
for rnd in range(SUPERDENSE_ROUNDS):
    msc.append_superdense_cycle(circuitry, rnd)
msc.append_cultivation(circuitry)
msc.append_teleportation(circuitry, TELEPORT_ROUNDS)
msc.append_expansion(circuitry)

msc.annotate_detectors(
    circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS
)

circuitry.append_observable(
    0,
    "Y_OBSERVABLE_EXPANDED",
    msc.target.logical(Pauli.Y, offset=TARGET_DISTANCE-5),
    *[
        "JCT0:Z0",
        "JCT0:Z1",
        "JCT0:Z2",
        "STN:TPT0:XB",
        "STN:TPT1:XB",
        "STN:TPT2:XB",
        "STN:DST:X1",
        "STN:DST:X5",
        "STN:DST:X6",
    ],
)

circuitry.detectors_report()

scenario = "Expanded"
scenarios[scenario] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.expanded")
point += 1

display(Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url()})"))

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Magic State Cultivation of $|\mathbf{S}\rangle$ [Corrected $\overline{\mathbf{Y}}$]"
sample(scenarios, title=title, label="Point", shots=1e5, correction=True, fontsize=10)

In [ ]:
simulate(
    scenarios,
    title,
    label="Point",
    postselection=True,
    shots=1e5,
    minimal_noise=-6,
    figsize=(11, 4.5),
    num_workers=7,
)